[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/floodplains_and_valleys/ohio_river_floodplain_widths.ipynb)

# Floodplains and valley widths along the Ohio River

In this notebook we map the floodplain of a stretch of the Ohio River valley, on the West Virginia–Ohio border, and measure how wide it is along every valley in the area.

The workflow is:
1. Install `lsdtt3` (the command-line analysis tools) and `lsdviztools3` (the plotting package).
2. Download a **10 m** USGS 3DEP DEM with `lsdtt-fetch-raster`.
3. Make a hillshade with `lsdtt-raster-preprocessing`.
4. Find the floodplain with `lsdtt-fluvial-surfaces`, using the method of Clubb et al. (2017): thresholds on HAND (height above nearest drainage) and gradient, **chosen automatically** from the data.
5. Measure the floodplain width on cross sections down every valley with `lsdtt-valley-centreline`.
6. Look at the widths: maps, a width profile, and a table.
7. Rerun the floodplain with **thresholds you choose yourself**, and compare.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

This step takes around 2 minutes. **`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now install `pygmt` (GMT is the Generic Mapping Tools, which `lsdviztools3` uses to make maps) and `matplotlib` (which `lsdviztools3` uses for the QQ plots and width profiles). This is the slowest step and takes around a minute.

In [ ]:
# Install pygmt. mamba prints hundreds of lines, so we hide them and only
# show the end of the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt", "matplotlib"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt and matplotlib installed successfully.")
else:
    print("pygmt/matplotlib installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Get lsdviztools3

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.3.0-py3-none-any.whl

In [ ]:
!pip install lsdviztools3-0.3.0-py3-none-any.whl

### Get the lsdtt3 command-line tools

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.8.0.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.8.0.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.8.0"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's older proj.db breaks rasterio and pyproj.

Check that it works: this should print version information (0.8.0).

In [ ]:
!lsdtt-fluvial-surfaces -v

## Step 1: Download a 10 m DEM

`lsdtt-fetch-raster` downloads a DEM for a box given in longitude and latitude. Here we use

* `dem_source: us_3dep`: the USGS 3D Elevation Program seamless DEM. It is a **bare-earth terrain model** (trees and buildings removed), which is what we want for floodplains. It covers the United States, needs no account or key, and its default product is **10 m** (1/3 arc-second). We do not set `grid_spacing`, so the output stays at the native 10 m.
* We do not set `target_epsg` either: the tool picks the UTM zone from the centre of the box (here UTM zone 17N, EPSG:32617), so distances are in metres.

The box (about 17 km east–west by 13 km north–south) takes in the Ohio River valley around Moundsville, West Virginia, with the tributary valleys that join it from both sides. At 10 m that is roughly 1700 by 1300 pixels.

The output is called `<write_fname>_DEM.tif`, so here `ohio_DEM.tif`.

### Reading the corners of your box

This cell defines two small functions that read latitude and longitude pairs copied from Google Maps. You do not need to edit it; just run it.

In [ ]:
import math
import re


def parse_lat_lon(text):
    """Read a "latitude, longitude" pair, as copied from Google Maps, into two numbers."""
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    if len(numbers) != 2:
        raise ValueError(
            f'Could not find a latitude and a longitude in "{text}". Right-click the place '
            'on Google Maps, click the two numbers at the top of the menu to copy them, and '
            'paste them between the quotes, for example "55.9476, -3.1582".')
    latitude = float(numbers[0])
    longitude = float(numbers[1])
    if not -90 <= latitude <= 90:
        raise ValueError(f'"{text}": the first number is the latitude and must be between '
                         '-90 and 90. Google Maps gives the latitude first.')
    if not -180 <= longitude <= 180:
        raise ValueError(f'"{text}": the second number is the longitude and must be '
                         'between -180 and 180.')
    return latitude, longitude


def box_from_corners(lower_left, upper_right):
    """Return west, east, south and north for the box with these two opposite corners."""
    lat_1, lon_1 = parse_lat_lon(lower_left)
    lat_2, lon_2 = parse_lat_lon(upper_right)
    if lat_1 == lat_2 or lon_1 == lon_2:
        raise ValueError("The two corners must have different latitudes and different longitudes.")
    # min and max, so you get the same box whichever pair of opposite corners you pick
    west, east = min(lon_1, lon_2), max(lon_1, lon_2)
    south, north = min(lat_1, lat_2), max(lat_1, lat_2)
    width_km = (east - west) * 111.32 * math.cos(math.radians((south + north) / 2))
    height_km = (north - south) * 110.57
    print(f"Box: west {west:.5f}, east {east:.5f}, south {south:.5f}, north {north:.5f}")
    print(f"Size: about {width_km:.0f} km east-west by {height_km:.0f} km north-south")
    return west, east, south, north

### Edit code block below for your site!

**Choosing the box with two right-clicks on Google Maps:**

1. On Google Maps, right-click the **lower left (south-west)** corner of your area. The first line of the menu that appears is the latitude and longitude. Click it to copy it.
2. Paste it between the quotes after `lower_left =`.
3. Do the same for the **upper right (north-east)** corner, after `upper_right =`.

`us_3dep` only covers the United States. Elsewhere, change `dem_source` to `cop30_aws` (30 m, global) and lower `threshold_contributing_pixels` in Steps 3 and 4 (see the table in Step 3).

In [ ]:
# Right-click the lower left (south-west) corner on Google Maps, copy, paste:
lower_left = "39.7962083281973, -80.90942932847119"
# ... and the upper right (north-east) corner:
upper_right = "39.91622439873636, -80.7047777293909"

west, east, south, north = box_from_corners(lower_left, upper_right)

In [ ]:
# Parameter file for lsdtt-fetch-raster
with open("ohio_fetch.param", "w") as f:
    f.write("write_fname: ohio\n")
    f.write(f"west: {west}\n")
    f.write(f"east: {east}\n")
    f.write(f"south: {south}\n")
    f.write(f"north: {north}\n")
    f.write("dem_source: us_3dep\n")   # USGS 3DEP, 10 m bare earth

!lsdtt-fetch-raster ./ ohio_fetch.param

In [ ]:
!ls -lh ohio*

## Step 2: Make a hillshade

A hillshade is our background map. `lsdtt-raster-preprocessing` writes `ohio_hillshade.tif`, lit from the northwest.

In [ ]:
with open("ohio_hillshade.param", "w") as f:
    f.write("read_fname: ohio_DEM.tif\n")
    f.write("write_fname: ohio\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")

!lsdtt-raster-preprocessing ./ ohio_hillshade.param

In [ ]:
from IPython.display import Image, display

from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.base import save_figure

hs_style = MapStyle(cmap="gray", transparent=False)
fig = render_hillshade("ohio_hillshade.tif", coordinates="local", shade=False, style=hs_style)
save_figure(fig, "ohio_hillshade.png", style=hs_style)
display(Image("ohio_hillshade.png", width=700))

## Step 3: Find the floodplain

### How it works (Clubb et al., 2017)

`lsdtt-fluvial-surfaces` with `fluvial_surface_algorithm_choice: clubb2017`:

1. builds a **channel network**, and computes for every pixel its **HAND**: the height above the channel pixel its water flows to (Nobre et al., 2011);
2. computes the **gradient** of every pixel (m/m: metres of rise per metre across), from a smooth surface fitted around it;
3. calls a pixel **floodplain** when both its HAND and its gradient are below a threshold;
4. keeps only floodplain patches that touch a channel, and fills their holes.

### Choosing the channel network

The network decides where HAND is measured from, and (in Step 4) which valleys get widths.

* A channel starts where the upslope area reaches `threshold_contributing_pixels`. This is a **count of pixels**, so the same area needs a different number at each resolution:

| Channels start at | 30 m DEM | 10 m DEM | 5 m DEM | 1 m DEM |
|---|---|---|---|---|
| 0.1 km² | 111 | 1000 | 4000 | 100000 |
| 1 km² | 1111 | 10000 | 40000 | 1000000 |

  We use **10000 pixels = 1 km²** on our 10 m DEM.
* `clubb2017` then counts a pixel as channel only if its **Strahler stream order** is at least `min_channel_stream_order` (3, as in Clubb et al. 2017), so small headwater streams do not grow floodplain of their own.

### Automatic thresholds: quantile–quantile plots

`clubb2017` chooses the HAND and gradient thresholds from the data (`floodplain_threshold_algorithm_choice: qq_gaussian`). For each quantity it makes a **quantile–quantile (QQ) plot** against a normal distribution, draws a straight reference line through the 25th and 75th percentiles, and takes the lowest value at which the data return to that line (within 1% of the data range) and stay on it. Hillslopes make up the bulk of the distribution; floodplain pixels form a low tail that departs from the line, and the threshold is where that tail ends.

On a big DEM the first-pass HAND threshold can be high, because wide flats above the valley floor (terraces, fans) are in the low tail too. `floodplain_second_pass_algorithm_choice: qq_upper_departure` (not part of Clubb et al. 2017) draws a second QQ plot over the first-pass floodplain only and lowers the HAND threshold to where that population departs upward. Set it to `none` to keep the first pass.

On very flat landscapes `lsdtt-fluvial-surfaces` may drop the gradient threshold and use HAND alone; the output tells you if it did (`floodplain_bound_algorithm_choice`).

In [ ]:
with open("ohio_FP.param", "w") as f:
    f.write("read_fname: ohio_DEM.tif\n")
    f.write("write_fname: ohio_FP\n")
    f.write("write_output_format: gtiff\n")

    # the channel network
    f.write("threshold_contributing_pixels: 10000\n")      # 1 km2 on a 10 m DEM

    # Clubb et al. (2017), automatic QQ thresholds, plus the second pass
    f.write("fluvial_surface_algorithm_choice: clubb2017\n")
    f.write("floodplain_second_pass_algorithm_choice: qq_upper_departure\n")

    # outputs
    f.write("write_floodplain_mask_raster: true\n")
    f.write("write_floodplain_polygons: true\n")
    f.write("write_reference_field_raster: true\n")       # HAND
    f.write("write_floodplain_gradient_raster: true\n")   # gradient

!lsdtt-fluvial-surfaces ./ ohio_FP.param

### The thresholds it chose

`ohio_FP_floodplain_thresholds.csv` lists every threshold, the pass it belongs to, and how many pixels it was read from. Always look at these before using the floodplain.

In [ ]:
import pandas as pd

thresholds = pd.read_csv("ohio_FP_floodplain_thresholds.csv")
thresholds

### The floodplain on the map, and the QQ plots

`lsdviz3-floodplain` (from `lsdviztools3`) draws the floodplain polygons over the hillshade, with HAND coloured underneath. When the second pass ran, the first-pass floodplain is drawn too. It also draws the HAND QQ plots the thresholds were read from.

In [ ]:
!lsdviz3-floodplain --stem ohio_FP --base ohio_hillshade.tif \
    --drape ohio_FP_reference_field.tif --drape-vmin 0 --drape-vmax 60 \
    --out-map ohio_floodplain_map.png --out-qq ohio_hand_qq.png

display(Image("ohio_floodplain_map.png", width=800))
display(Image("ohio_hand_qq.png", width=800))

**Question:** does the mapped floodplain follow the flat valley floor you can see in the hillshade? Where does it climb onto terraces, and where does it miss valley floor?

## Step 4: Measure the valley widths

`lsdtt-valley-centreline` takes the floodplain mask from Step 3 (`valley_mask_fname`). With `run_valley_width_network: true` it works on **every** channel:

1. Channels are traced downstream from their heads, longest first. Each one stops where it meets a channel already traced, so the trunk of each basin is its longest flow path, every tributary ends at its parent, and each valley is measured once.
2. A **centreline** is drawn down the middle of each valley floor (`network_centreline_algorithm_choice: medial_path`), so the cross sections do not swing with every meander of the river.
3. **Cross sections** are cast across the centreline every `cross_section_spacing_metres`. The width at each station is the span of floodplain the section crosses.

Use **the same channel network as Step 3**: the same `threshold_contributing_pixels`, and the same stream-order rule (this tool's default is every channel above the threshold, so we set `stream_order` here).

In [ ]:
with open("ohio_W.param", "w") as f:
    f.write("read_fname: ohio_DEM.tif\n")
    f.write("write_fname: ohio_W\n")
    f.write("write_output_format: gtiff\n")

    # the floodplain from Step 3
    f.write("valley_mask_fname: ohio_FP_floodplain_mask.tif\n")

    # the SAME channel network as Step 3
    f.write("threshold_contributing_pixels: 10000\n")
    f.write("channel_definition_algorithm_choice: stream_order\n")
    f.write("min_channel_stream_order: 3\n")

    # every channel, centrelines and cross sections
    f.write("run_valley_width_network: true\n")
    f.write("network_centreline_algorithm_choice: medial_path\n")
    f.write("cross_section_spacing_metres: 200\n")
    f.write("write_cross_section_features: false\n")   # not needed for widths, and slow

!lsdtt-valley-centreline ./ ohio_W.param

## Step 5: Look at the widths

`ohio_W_valley_widths.csv` has one row per cross section. The important columns:

| Column | Meaning |
|---|---|
| `segment_id`, `parent_segment_id` | Which channel segment, and the one it flows into (-1 for a trunk) |
| `along_distance_m` | Distance down the centreline |
| `valley_width_m` | **Floodplain width** (m) |
| `drainage_area_m2` | Drainage area at the station |
| `catchment_touches_edge` | 1 if the catchment is cut by the edge of the DEM |
| `excluded_reason` | `none` = a valid width. Otherwise why the section was left out |

The reasons a section is left out:

| `excluded_reason` | Meaning |
|---|---|
| `not_in_valley` | The channel runs outside the floodplain here (a steep reach with no floodplain). Usually the most common |
| `near_parent_channel` | A tributary's section inside its parent's valley: the parent measures it |
| `tributary_crossing` | A parent's section that runs up a tributary valley, which would read too wide |
| `side_not_found` | No floodplain edge found within the section |

In [ ]:
widths = pd.read_csv("ohio_W_valley_widths.csv")
print("Sections by reason:")
print(widths["excluded_reason"].value_counts())

kept = widths[(widths["excluded_reason"] == "none") & (widths["valley_width_m"] > 0)]
print(f"\n{len(kept)} valid widths. Summary (m):")
print(kept["valley_width_m"].describe().round(1))

The Ohio River itself enters and leaves across the edge of the DEM, so its `catchment_touches_edge` is 1: its drainage area here is only the part inside the box. Its widths are still real; only the drainage area is incomplete.

Median width of each valley segment, widest first:

In [ ]:
per_segment = (kept.groupby("segment_id")
                   .agg(n_sections=("valley_width_m", "size"),
                        median_width_m=("valley_width_m", "median"),
                        max_drainage_area_km2=("drainage_area_m2", lambda a: a.max() / 1e6))
                   .sort_values("median_width_m", ascending=False))
per_segment.round(1).head(15)

### Maps and a width profile

`lsdviz3-valley-widths` draws:

* the cross sections over the hillshade, coloured by `excluded_reason`, with the floodplain mask underneath;
* the floodplain width down the trunk with the largest drainage area (here the Ohio River), with the tributary junctions marked;
* each segment's channel (blue) and valley centreline (orange).

In [ ]:
!lsdviz3-valley-widths --stem ohio_W --base ohio_hillshade.tif \
    --valley-mask ohio_FP_floodplain_mask.tif \
    --out-map ohio_widths_map.png --out-profile ohio_widths_profile.png \
    --out-channels ohio_widths_channels.png

display(Image("ohio_widths_map.png", width=800))
display(Image("ohio_widths_profile.png", width=800))
display(Image("ohio_widths_channels.png", width=800))

A simple plot of width against drainage area, made with matplotlib from the table:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(kept["drainage_area_m2"] / 1e6, kept["valley_width_m"], s=8, alpha=0.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Drainage area (km²)")
ax.set_ylabel("Floodplain width (m)")
ax.set_title("Floodplain width against drainage area")
plt.show()

## Step 6: Setting the thresholds yourself

The automatic thresholds are a statistical estimate. If the floodplain spills up side slopes, or misses valley floor you can see, set the thresholds yourself:

* `floodplain_threshold_algorithm_choice: fixed` switches off the QQ method;
* `max_floodplain_hand_metres`: HAND (m) above which ground is not floodplain;
* `max_floodplain_gradient`: gradient (m/m, **not degrees**) above which ground is not floodplain. 0.05 is about 3°, 0.1 about 6°;
* `floodplain_bound_algorithm_choice: hand_and_gradient` uses both thresholds, as Clubb et al. (2017) did (`hand_only` uses HAND alone).

A good way to choose them: compare the HAND map (`ohio_FP_reference_field.tif`) and the gradient map (`ohio_FP_floodplain_gradient.tif`) with the hillshade, and read off their values at the edge of the valley floor. Here we try **HAND 10 m and gradient 0.05**. Change them and rerun these cells.

In [ ]:
max_hand = 10       # metres
max_gradient = 0.05 # m/m

with open("ohio_FPmanual.param", "w") as f:
    f.write("read_fname: ohio_DEM.tif\n")
    f.write("write_fname: ohio_FPmanual\n")
    f.write("write_output_format: gtiff\n")
    f.write("threshold_contributing_pixels: 10000\n")
    f.write("fluvial_surface_algorithm_choice: clubb2017\n")
    f.write("floodplain_threshold_algorithm_choice: fixed\n")   # overrides the automatic QQ thresholds
    f.write(f"max_floodplain_hand_metres: {max_hand}\n")
    f.write(f"max_floodplain_gradient: {max_gradient}\n")
    f.write("floodplain_bound_algorithm_choice: hand_and_gradient\n")
    f.write("floodplain_second_pass_algorithm_choice: none\n")
    f.write("write_floodplain_mask_raster: true\n")
    f.write("write_floodplain_polygons: true\n")
    f.write("write_reference_field_raster: true\n")

!lsdtt-fluvial-surfaces ./ ohio_FPmanual.param

Now measure the widths against this floodplain. The parameter file is the one from Step 4 with a different mask and output name.

In [ ]:
with open("ohio_W.param") as f:
    text = f.read()
text = text.replace("ohio_FP_floodplain_mask.tif", "ohio_FPmanual_floodplain_mask.tif")
text = text.replace("write_fname: ohio_W", "write_fname: ohio_Wmanual")
with open("ohio_Wmanual.param", "w") as f:
    f.write(text)

!lsdtt-valley-centreline ./ ohio_Wmanual.param

In [ ]:
!lsdviz3-floodplain --stem ohio_FPmanual --base ohio_hillshade.tif \
    --drape ohio_FPmanual_reference_field.tif --drape-vmin 0 --drape-vmax 60 \
    --out-map ohio_floodplain_manual_map.png
display(Image("ohio_floodplain_manual_map.png", width=800))

manual = pd.read_csv("ohio_Wmanual_valley_widths.csv")
kept_manual = manual[(manual["excluded_reason"] == "none") & (manual["valley_width_m"] > 0)]

comparison = pd.DataFrame({
    "automatic (QQ)": kept["valley_width_m"].describe(),
    f"manual (HAND {max_hand} m, gradient {max_gradient})": kept_manual["valley_width_m"].describe(),
}).round(1)
comparison

## Step 7: Measuring only the rivers you choose

So far every channel above the threshold was measured. To measure only particular rivers, give both tools a file of **source points**: a CSV with `latitude` and `longitude` columns (WGS84, as copied from Google Maps), one row per river, placed near where you want each river to start. Each point is snapped onto the nearest channel and traced downstream to the edge of the DEM.

* In both parameter files add `source_points_fname: ohio_sources.csv`.
* In the `lsdtt-valley-centreline` file **remove** `run_valley_width_network: true` (it measures every channel, and refuses a sources file).
* With sources, the widths are on the section lines (`ohio_W_valley_cross_sections.fgb`, field `valley_width_m`) rather than in `_valley_widths.csv`.
* Two sources whose rivers join share the river below the junction, so it is measured once for each of them.

```python
pd.DataFrame({"latitude": [39.85], "longitude": [-80.80]}).to_csv("ohio_sources.csv", index=False)
```

## Things to try

* Change `threshold_contributing_pixels` (for example to 3000 or 30000) in **both** Step 3 and Step 4. Which valleys gain or lose widths?
* Set `floodplain_second_pass_algorithm_choice: none` in Step 3 and compare the thresholds and the map.
* Change `network_centreline_algorithm_choice` to `smoothed_channel` in Step 4 (the river's own course, smoothed) and compare the widths.
* Change `cross_section_spacing_metres` to 100 for more sections.
* Move the box to another valley in the United States. Outside the US, use `dem_source: cop30_aws` (30 m) and divide the thresholds by 9.

## Every run is recorded

Each program writes `<write_fname>_ingested_params.param`: every parameter it used, including the thresholds it chose for itself. You can run it again as it is to repeat the run. Keep these files with your results.

## References

* Clubb, F. J., Mudd, S. M., Milodowski, D. T., Valters, D. A., Slater, L. J., Hurst, M. D. and Limaye, A. B. (2017). Geomorphometric delineation of floodplains and terraces from objectively defined topographic thresholds. *Earth Surface Dynamics*, 5, 369–385. https://doi.org/10.5194/esurf-5-369-2017
* Nobre, A. D., Cuartas, L. A., Hodnett, M., Rennó, C. D., Rodrigues, G., Silveira, A., Waterloo, M. and Saleska, S. (2011). Height Above the Nearest Drainage: a hydrologically relevant new terrain model. *Journal of Hydrology*, 404, 13–29. https://doi.org/10.1016/j.jhydrol.2011.03.051